# Case Study §4 — CPT from BASE vs INSTRUCT, and catastrophic forgetting

Runnable twin of [`04_cpt_base_vs_instruct.py`](04_cpt_base_vs_instruct.py). Research question #4:
does the starting point (base vs instruct) matter for CPT, and does CPT on the domain corpus damage
general ability?

**What we measure** (before & after CPT, for each starting model):
- **domain** perplexity on held-out chemistry text → should **drop** (learning the domain)
- **general** perplexity on held-out non-chemistry text → if it **rises**, that's forgetting
- a few **general question** generations → eyeball whether instruction-following degraded

The instruct model has more general skill to lose; the base model has little. Mitigations (lower LR,
≤1 epoch, ~10% replay) are in `PITFALLS.md`.

> `MODE="trial"` validates plumbing in seconds (deltas ≈ 0 — expected). Set `"full"` for real effects.

In [ ]:
MODE = "trial"     # "trial" or "full"
FORCE = False
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = next(p for p in [HERE, HERE/'scripts', HERE.parent/'scripts'] if (p/'config.py').exists())
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s4", root / "04_cpt_base_vs_instruct.py")
s4 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s4)
print(f"mode={config.RUN_MODE}")

## Run the experiment
Self-sufficient (builds the domain corpus + fetches a small general corpus if missing, downloads both
SmolLM2-135M and -Instruct) and idempotent (cached to `outputs/forgetting_<mode>.json`). It prints
general-question answers **before and after** CPT for each model — eyeball them for forgetting.

In [ ]:
m = s4.run(force=FORCE)
print(json.dumps(m['summary'], indent=2))

## Verify & read the result
Domain Δ negative = learned the domain; general Δ positive = forgetting. (In TRIAL both are ≈0 — only a
few steps. Run `MODE="full"` for the real contrast.)

In [ ]:
for k in ['base_domain_delta_pct','base_general_delta_pct','instruct_domain_delta_pct','instruct_general_delta_pct']:
    assert k in m['summary'], k
for who in ('base','instruct'):
    r = m[who]
    assert r['domain_ppl_before'] > 0 and r['general_ppl_before'] > 0
print('\u2713 §4 verified: domain + general perplexity measured before/after for base and instruct.')
print('Next: \u00a75 SFT (completion-only loss) on the small Q&A set, then \u00a76 the base-vs-instruct sweep.')